In [63]:
pip install yfinance langchain langchain-openai langchain_openai langchain_core pydantic

In [69]:

# ============================================================
# Imports
# ============================================================

from typing import List
from pydantic import BaseModel

import yfinance as yf

from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.tools import StructuredTool
from langchain_core.prompts import PromptTemplate


# ============================================================
# 1. Pydantic Models
# ============================================================

class StockItem(BaseModel):
    symbol: str
    price: float


class StockPriceList(BaseModel):
    stocks: List[StockItem]


# ============================================================
# 2. Stock Price Function
# ============================================================

def get_stock_prices(symbols: List[str]) -> StockPriceList:
    """Get the latest stock prices for multiple ticker symbols."""

    stocks = []

    for symbol in symbols:

        data = yf.Ticker(symbol).history(period="1d")

        if data.empty:
            continue

        price = float(data["Close"].iloc[-1])

        stocks.append(
            StockItem(
                symbol=symbol.upper(),
                price=round(price, 2)
            )
        )

    return StockPriceList(stocks=stocks)


# ============================================================
# 3. Create Tool
# ============================================================

stock_price_tool = StructuredTool.from_function(
    func=get_stock_prices,
    name="get_stock_prices",
    description="""
    Get the latest stock prices for one or more ticker symbols.
    Input must be a list of ticker symbols.

    Example:
    ["AAPL", "AMZN", "MSFT"]
    """
)


# ============================================================
# 4. Create Prompt
# ============================================================

prompt_template = PromptTemplate.from_template("""
You are a financial assistant with access to the get_stock_prices tool.

Your job is to answer user requests about stock prices.

Rules:

1. Identify ALL companies mentioned by the user.

2. Convert every company name to its correct ticker symbol.

3. When the user asks for stock prices,
   ALWAYS call the get_stock_prices tool.

4. Call get_stock_prices only ONCE.

5. Pass ALL ticker symbols as a list.

6. Never guess stock prices yourself.

7. Do not return only ticker symbols.

Examples:

Apple -> AAPL
Amazon -> AMZN
Microsoft -> MSFT
Tesla -> TSLA
Nvidia -> NVDA
Google -> GOOGL
Meta -> META

Example user request:

قیمت اپل آمازون مایکروسافت

Expected tool call:

get_stock_prices(
    symbols=["AAPL", "AMZN", "MSFT"]
)

User request:

{user_input}
""")


# ============================================================
# 5. Create LLM
# ============================================================

api_key = userdata.get("OpenRouter")

client = ChatOpenAI(
    model="openai/gpt-5-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)


# ============================================================
# 6. Bind Tool To LLM
# ============================================================

client_with_tools = client.bind_tools([
    stock_price_tool
])


# ============================================================
# 7. Create Chain
# ============================================================

chain = prompt_template | client_with_tools


# ============================================================
# 8. User Request
# ============================================================

user_request = input("Enter your request: ")


# ============================================================
# 9. LLM
# ============================================================

response = chain.invoke({
    "user_input": user_request
})


# ============================================================
# 10. Execute Tool
# ============================================================

if response.tool_calls:

    for tool_call in response.tool_calls:

        print("Tool Name:", tool_call["name"])
        print("Tool Arguments:", tool_call["args"])

        # Check which tool LLM requested
        if tool_call["name"] == "get_stock_prices":

            result = stock_price_tool.invoke(
                tool_call["args"]
            )

            for stock in result.stocks:
                print(
                    f"نماد {stock.symbol} قیمتش ${stock.price} است"
                )

        else:
            print(f"Unknown tool: {tool_call['name']}")

else:
    print(response.content)


Enter your request: بیتکوین امازون و ماکروسافت
Tool Name: get_stock_prices
Tool Arguments: {'symbols': ['BTC-USD', 'AMZN', 'MSFT']}
نماد BTC-USD قیمتش $75763.76 است
نماد AMZN قیمتش $248.42 است
نماد MSFT قیمتش $497.12 است
